# Laboratorio en Google Colab: Agentes de IA con Groq

Versión adaptada para evitar conflictos de dependencias en Google Colab.

**Idea central:** usar Groq como proveedor LLM y comparar varias formas de construir agentes:

1. Llamada directa a Groq.
2. Agente simple implementado desde cero.
3. CrewAI.
4. LangGraph.
5. AutoGen.
6. LlamaIndex.

## 0. Configuración de la API key de Groq

In [12]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Ingresa GROQ_API_KEY: ")

# Seleccionamos un modelo de los que Groq dispone
MODEL = "openai/gpt-oss-20b"

print("GROQ_API_KEY configurada:", bool(os.environ.get("GROQ_API_KEY")))

GROQ_API_KEY configurada: True


## 1. Instalación mínima recomendada



In [1]:
!pip -q install --upgrade groq python-dotenv crewai>=0.130.0 crewai-tools>=0.47.0 litellm>=1.70.0
# Si presenta una falla, reiniciar el entorno y volver a correr la celda

import groq
print("Groq instalado correctamente")

Groq instalado correctamente


## 2. Llamada directa a Groq

Antes de usar frameworks de agentes, conviene verificar que el modelo responde.

In [3]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Eres un asistente claro y didáctico."},
        {"role": "user", "content": "Explica en 3 líneas qué es un agente de IA."},
    ],
    temperature=0.2,
)

print(response.choices[0].message.content)

Un agente de IA es un programa que percibe su entorno a través de sensores (datos de entrada) y actúa sobre él mediante actuadores (salidas).  
Utiliza algoritmos de aprendizaje o reglas predefinidas para decidir la mejor acción que maximice un objetivo o recompensa.  
En esencia, combina percepción, razonamiento y acción para interactuar de forma autónoma con su entorno.


## 3. Agente simple desde cero

Un agente no es solo un chatbot. Normalmente combina:

- un objetivo,
- memoria o estado,
- herramientas,
- razonamiento sobre qué acción tomar,
- ejecución de acciones.

In [4]:
from typing import Callable, Dict

def calculator(expression: str) -> str:
    """Calculadora simple para expresiones aritméticas seguras."""
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Expresión no permitida."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

tools: Dict[str, Callable[[str], str]] = {
    "calculator": calculator
}

def ask_groq(messages, temperature=0.2):
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

def simple_agent(user_question: str) -> str:
    system = """
Eres un agente con una herramienta llamada calculator.
Si necesitas calcular, responde exactamente en este formato:
TOOL: calculator
INPUT: expresión
Si no necesitas herramienta, responde directamente.
"""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_question},
    ]
    first = ask_groq(messages)
    print("--- Decisión del agente ---")
    print(first)

    if "TOOL: calculator" in first and "INPUT:" in first:
        expression = first.split("INPUT:", 1)[1].strip()
        tool_result = calculator(expression)
        messages.append({"role": "assistant", "content": first})
        messages.append({"role": "user", "content": f"Resultado de la herramienta: {tool_result}. Responde al usuario de forma final."})
        return ask_groq(messages)
    return first

print(simple_agent("Si tengo 18 estudiantes y hago grupos de 3, ¿cuántos grupos salen?"))

--- Decisión del agente ---
Hay 6 grupos.
Hay 6 grupos.


# Plataformas de agentes



## 4. CrewAI con Groq

CrewAI es útil para ejemplos tipo “equipo de trabajo”: investigador, analista, redactor, revisor, etc.

In [14]:
from crewai import Agent, Task, Crew, Process, LLM
import crewai.llms.cache as _crewai_cache

# Parche para funcionar en COLAB
_crewai_cache.mark_cache_breakpoint = lambda msg: msg
# Fuente: https://github.com/crewAIInc/crewAI/issues/5886?
# Curso free:

llm = LLM(
    model=f"groq/{MODEL}",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.2,
)

investigador = Agent(
    role="Investigador de IA",
    goal="Encontrar ideas claras para enseñar agentes de IA",
    backstory="Eres un profesor experto en IA aplicada y educación universitaria.",
    llm=llm,
    verbose=True,
)

redactor = Agent(
    role="Redactor académico",
    goal="Convertir ideas técnicas en una explicación didáctica",
    backstory="Eres especialista en comunicación científica clara.",
    llm=llm,
    verbose=True,
)

tarea1 = Task(
    description="Propón 5 conceptos clave para explicar agentes de IA a estudiantes de computación.",
    expected_output="Lista de 5 conceptos con breve explicación.",
    agent=investigador,
)

tarea2 = Task(
    description="Convierte esos conceptos en una mini actividad de clase de 20 minutos.",
    expected_output="Actividad con objetivo, dinámica y entregable.",
    agent=redactor,
)

crew = Crew(
    agents=[investigador, redactor],
    tasks=[tarea1, tarea2],
    process=Process.sequential,
    verbose=True,
)

resultado = await crew.kickoff_async()

print(resultado)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 409e4c1d-e31b-424c-a0c3-eb0b63e8edbe                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Propón 5 conceptos clave para explicar agentes de IA a estudiantes de computación.                       │
│  ID: d313c645-4a2a-4a3b-9f43-f43d6ef4fd97                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador de IA                                                                                      │
│                                                                                                                 │
│  Task: Propón 5 conceptos clave para explicar agentes de IA a estudiantes de computación.                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador de IA                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. **Percepción** – El mecanismo por el cual un agente recibe información del entorno a través de sensores     │
│  (sensores físicos, APIs, archivos, etc.) y la convierte en datos procesables.                                  │
│  2. **Representación del conocimiento** – La forma en que el agente almacena y organiza la información          │
│  (modelos, grafos, bases de datos, redes neuronales) para poder razonar y tomar decisiones.                     │
│  3. **Planificación y toma de decisiones** – Los algoritmos que permiten al agente elegir acciones (búsqueda,   │
│  optimización, aprendizaje por refuerzo) para alcanzar objetivos definidos.                                     │
│  4. **Aprendizaje** – La capacidad de mejorar su comportamiento con experiencia, ya sea mediante aprendizaje    │
│  supervisado, no supervisado o por refuerzo, adaptándose a cambios en el entorno.                               │
│  5. **Interacción y comunicación** – Los protocolos y interfaces que permiten al agente comunicarse con         │
│  usuarios o con otros agentes (mensajes, diálogos, APIs), facilitando la colaboración y la coordinación en      │
│  sistemas multiagente.                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor académico                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ## Mini‑actividad de 20 minutos                                                                                │
│  **Tema:** “Construyendo un agente inteligente paso a paso”                                                     │
│  **Nivel:** Estudiantes de primer año de Ingeniería o Ciencias de la Computación                                │
│  **Duración total:** 20 minutos                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Objetivo de la actividad                                                                                │
│  Que los estudiantes comprendan de forma práctica y visual cómo se interrelacionan los cinco componentes clave  │
│  de un agente inteligente (percepción, representación del conocimiento, planificación y toma de decisiones,     │
│  aprendizaje, interacción y comunicación) y que diseñen, en equipo, un esquema simplificado de un agente que    │
│  resuelva un problema concreto.                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Materiales necesarios                                                                                   │
│  - Pizarra o rotafolios y rotuladores (o una herramienta digital colaborativa como Miro, Jamboard, Google       │
│  Jamboard).                                                                                                     │
│  - Tarjetas de papel (o fichas digitales) con los nombres de los cinco componentes.                             │
│  - Tarjetas de papel con ejemplos de “problemas” (p. ej. “organizar la agenda de un estudiante”, “controlar la  │
│  temperatura de un cuarto”, “jugar a Tic‑Tac‑Toe”).                                                             │
│  - Reloj o temporizador visible.                                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 3. Dinámica paso a paso                                                                                    │
│                                                                                                                 │
│  | Tiempo | Paso | Descripción | Responsables |                                                                 │
│  |--------|------|-------------|--------------|                                                                 │
│  | **0‑2 min** | **Introducción rápida** | El docente e

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Convierte esos conceptos en una mini actividad de clase de 20 minutos.                                   │
│  Agent: Redactor académico                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 409e4c1d-e31b-424c-a0c3-eb0b63e8edbe                                                                       │
│  Final Output: ## Mini‑actividad de 20 minutos                                                                  │
│  **Tema:** “Construyendo un agente inteligente paso a paso”                                                     │
│  **Nivel:** Estudiantes de primer año de Ingeniería o Ciencias de la Computación                                │
│  **Duración total:** 20 minutos                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 1. Objetivo de la actividad                                                                                │
│  Que los estudiantes comprendan de forma práctica y visual cómo se interrelacionan los cinco componentes clave  │
│  de un agente inteligente (percepción, representación del conocimiento, planificación y toma de decisiones,     │
│  aprendizaje, interacción y comunicación) y que diseñen, en equipo, un esquema simplificado de un agente que    │
│  resuelva un problema concreto.                                                                                 │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 2. Materiales necesarios                                                                                   │
│  - Pizarra o rotafolios y rotuladores (o una herramienta digital colaborativa como Miro, Jamboard, Google       │
│  Jamboard).                                                                                                     │
│  - Tarjetas de papel (o fichas digitales) con los nombres de los cinco componentes.                             │
│  - Tarjetas de papel con ejemplos de “problemas” (p. ej. “organizar la agenda de un estudiante”, “controlar la  │
│  temperatura de un cuarto”, “jugar a Tic‑Tac‑Toe”).                                                             │
│  - Reloj o temporizador visible.                                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### 3. Dinámica paso a paso                                                                                    │
│                                                                                                                 │
│  | Tiempo | Paso | Descripción | Responsables |                                                                 │
│  |--------|------|-------------|--------------|                                                                 │
│  | **0‑2 min** | **Introducción rápida** | El docente 

/tmp/ipykernel_3336/81577484.py:49: RuntimeWarning: coroutine 'Crew.kickoff_async' was never awaited
  resultado = await crew.kickoff_async()


## Mini‑actividad de 20 minutos  
**Tema:** “Construyendo un agente inteligente paso a paso”  
**Nivel:** Estudiantes de primer año de Ingeniería o Ciencias de la Computación  
**Duración total:** 20 minutos  

---

### 1. Objetivo de la actividad  
Que los estudiantes comprendan de forma práctica y visual cómo se interrelacionan los cinco componentes clave de un agente inteligente (percepción, representación del conocimiento, planificación y toma de decisiones, aprendizaje, interacción y comunicación) y que diseñen, en equipo, un esquema simplificado de un agente que resuelva un problema concreto.

---

### 2. Materiales necesarios  
- Pizarra o rotafolios y rotuladores (o una herramienta digital colaborativa como Miro, Jamboard, Google Jamboard).  
- Tarjetas de papel (o fichas digitales) con los nombres de los cinco componentes.  
- Tarjetas de papel con ejemplos de “problemas” (p. ej. “organizar la agenda de un estudiante”, “controlar la temperatura de un cuarto”, “jugar a Tic‑Tac‑

## 5. LangGraph con Groq

LangGraph es recomendable cuando necesitas controlar explícitamente el flujo del agente como un grafo de estados.

In [15]:
# Instalación opcional para LangGraph. Ejecuta solo si usarás esta sección.
!pip -q install --upgrade langgraph langchain-core langchain-groq
print("LangGraph instalado")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.0 MB/s eta 0:00:00
LangGraph instalado


In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

llm_lg = ChatGroq(
    model=MODEL,
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.2, # Es la "creatividad" del modelo
)

class AgentState(TypedDict):
    pregunta: str
    respuesta: str

def responder(state: AgentState):
    messages = [
        SystemMessage(content="Eres un agente educativo experto en IA."),
        HumanMessage(content=state["pregunta"]),
    ]
    result = llm_lg.invoke(messages)
    return {"respuesta": result.content}

graph = StateGraph(AgentState)
graph.add_node("responder", responder)
graph.set_entry_point("responder")
graph.add_edge("responder", END)
app = graph.compile()

resultado = app.invoke({"pregunta": "Diseña una actividad breve para explicar agentes reactivos y deliberativos."})
print(resultado["respuesta"])

**Actividad breve: “El juego de los dos agentes”**  
*(Duración aproximada: 20‑25 min)*  

---

### Objetivo  
Que los estudiantes comprendan la diferencia entre **agentes reactivos** (responden directamente a estímulos) y **agentes deliberativos** (planifican antes de actuar) mediante una dinámica práctica y visual.

### Materiales  
- Dos fichas de colores (por ejemplo, una roja y una azul).  
- Un tablero simple con casillas numeradas del 1 al 10 (puede ser papel o una pizarra).  
- Tarjetas con “situaciones” (por ejemplo: “Hay un obstáculo en la casilla 5”, “El objetivo está en la casilla 8”, “El camino está bloqueado”).  
- Reloj o cronómetro (para limitar el tiempo de respuesta).  

### Preparación  
1. **Definir roles**:  
   - **Agente Reactivo**: Actúa inmediatamente según la información que recibe, sin planear.  
   - **Agente Deliberativo**: Recibe la misma información pero primero “planifica” una ruta antes de moverse.  

2. **Asignar fichas**: Cada estudiante o pareja reci

## 6. AutoGen con Groq

AutoGen es útil para conversaciones multiagente. Groq puede usarse mediante un cliente compatible con OpenAI usando `base_url`.

In [ ]:
# Instalación opcional para AutoGen. Ejecuta solo si usarás esta sección.
!pip -q install --upgrade autogen-agentchat autogen-ext[openai]
print("AutoGen instalado")

AutoGen instalado


In [ ]:
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model=MODEL,
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": True,
        "structured_output": True,
        "family": "llama",
    },
)

assistant = AssistantAgent(
    name="profesor_ia",
    model_client=model_client,
    system_message="Eres un profesor universitario que explica agentes de IA con ejemplos prácticos.",
)

async def run_autogen():
    response = await assistant.on_messages(
        [TextMessage(content="Dame un ejemplo de sistema multiagente para investigación académica.", source="user")],
        cancellation_token=CancellationToken(),
    )
    print(response.chat_message.content)

await run_autogen()

Un ejemplo de sistema multiagente para investigación académica es un sistema de cooperación entre agentes para realizar la búsqueda de información y la evaluación de artículos científicos.

**Ejemplo:**

Considere un sistema de investigación en el que hay tres agentes:

1. **Agente de Búsqueda de Información (IB)**: tiene como objetivo buscar artículos científicos relevantes a un tema específico. Utiliza un motor de búsqueda como Google Académico o PubMed para obtener una lista de artículos.
2. **Agente de Evaluación (EV)**: evalúa la calidad de los artículos buscados por el IB. Utiliza métricas como el índice de impacto, el número de citas o la revisión por pares para calificar los artículos.
3. **Agente de Resumen (RS)**: resume los artículos evaluados por el EV. Genera un resumen breve de cada artículo, destacando los puntos principales y las conclusiones.

**Ejemplo de comportamiento:**

* El IB busca artículos sobre el tema de "Aprendizaje Automático" y obtiene una lista de 100 ar

## 7. LlamaIndex con Groq

LlamaIndex es especialmente útil cuando el agente debe consultar documentos, notas, PDFs o bases de conocimiento.

In [16]:
# Instalación opcional para LlamaIndex. Ejecuta solo si usarás esta sección.
!pip -q install --upgrade llama-index llama-index-llms-groq
print("LlamaIndex instalado")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.6/173.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 12.2 MB/s eta 0:00:00
LlamaIndex instalado


In [ ]:
from llama_index.llms.groq import Groq as LlamaGroq
from llama_index.core.llms import ChatMessage

llm_li = LlamaGroq(
    model=MODEL,
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.2,
)

messages = [
    ChatMessage(role="system", content="Eres un asistente experto en agentes de IA."),
    ChatMessage(role="user", content="Explica cuándo usarías LlamaIndex en un agente basado en RAG."),
]

response = llm_li.chat(messages)
print(response.message.content)

LlamaIndex es una herramienta de búsqueda de conocimiento desarrollada por Meta AI, que se utiliza para mejorar la precisión y la eficiencia de los modelos de lenguaje basados en RAG (Retrieval-Augmented Generation). Un agente basado en RAG utiliza una combinación de búsqueda de conocimiento y generación de texto para responder a preguntas o realizar tareas.

Aquí hay algunas situaciones en las que usaría LlamaIndex en un agente basado en RAG:

1. **Mejora de la precisión**: LlamaIndex puede ayudar a mejorar la precisión de las respuestas del agente al proporcionar información más precisa y actualizada. Esto es especialmente útil en situaciones en las que la información es compleja o cambia con frecuencia.
2. **Reducción del tiempo de respuesta**: LlamaIndex puede acelerar el proceso de búsqueda de conocimiento, lo que reduce el tiempo de respuesta del agente. Esto es especialmente útil en aplicaciones que requieren respuestas rápidas, como asistentes virtuales o chatbots.
3. **Mejora 